In [ ]:
from dotenv import load_dotenv
import os
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

load_dotenv()
os.environ["LANGCHAIN_PROJECT"] = "RAG TUTORIAL"

LOCAL_MODEL_NAME = "qwen3-8b"
LOCAL_BASE_URL = "http://202.31.200.130:8001/v1"
LOCAL_API_KEY = "not_used"
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body",
                             "media_end_head_title"]},
        )
    ),
)
docs = loader.load()
print(f"문서의 수: {len(docs)}")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

splits = text_splitter.split_documents(docs)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# rlm/rag-prompt 동일 내용을 직접 정의 (hub 불필요)
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question}\nContext: {context}\nAnswer:"
)

# 로컬 qwen3-8b 모델 사용
llm = ChatOpenAI(
    model_name=LOCAL_MODEL_NAME,
    base_url=LOCAL_BASE_URL,
    api_key=LOCAL_API_KEY,
    temperature=0,
)

def format_docs(docs):
    return \"\n\n\".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke(
    "부영그룹의 출산 장려 정책에 대해 설명해주세요."
)

In [ ]:
rag_chain.invoke(
    "부영그룹에 대해 설명해주세요."
)